# IMAD Phase 1 — GenImage ResNet-18 experiment
This notebook uses the repository's GenImage-specific parser and training script. It does not download GenImage. First obtain one permitted generator archive manually from the [official GenImage repository](https://github.com/GenImage-Dataset/GenImage), read its dataset terms, and extract it while preserving `<generator>/{train,val}/{ai,nature}/...`. The dataset is restricted to noncommercial research/academic use under CC BY-NC-SA 4.0 plus additional terms.

The experiment maps `ai → fake`, `nature → real`. `nature` means collected from ImageNet in this benchmark, not proof of authenticity. If multiple generator folders with adequate class coverage are present and there is no official test, one complete generator is held out for internal test and the other generators' official train/val remain train/validation. With a single-generator archive, `val` is validation and an internal test is carved from official train.

Run the metadata/sample cells first. Full hashing can take substantial time. Then run a 1-epoch train/validation smoke fit before full training; the smoke path never evaluates internal test. Smoke metrics are not research results.

In [ ]:
import sys, subprocess, platform, json, os
print(sys.version, platform.platform())
import torch, torchvision
print('PyTorch', torch.__version__, 'torchvision', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select Runtime → Change runtime type → GPU before full training. CPU is suitable only for pipeline/smoke checks.')


## Get the project code into Colab
Commit and push the reviewed repository changes first, then this clones the repository's current `main` branch. The local Codex workspace and Colab runtime do not share filesystems.


In [ ]:
PROJECT_URL = 'https://github.com/ronitjais/IMAD.git'
PROJECT_ROOT = '/content/IMAD'
if not os.path.isdir(PROJECT_ROOT):
    subprocess.run(['git', 'clone', PROJECT_URL, PROJECT_ROOT], check=True)
os.chdir(PROJECT_ROOT)
# Avoid replacing Colab's CUDA-matched torch/torchvision stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
from pathlib import Path
missing = []
for package in ('PIL', 'matplotlib'):
    try: __import__(package)
    except ImportError: missing.append('Pillow' if package == 'PIL' else package)
if missing: subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import PIL, matplotlib
print('Project:', Path.cwd(), '| Pillow', PIL.__version__, '| matplotlib', matplotlib.__version__)


## Mount storage and choose an already extracted dataset
Open the official dataset page, follow its Google Drive or Baidu archive link, and download only the generator subset you intend to study. The README gives Baidu code `ztf1`. Extract into Drive and preserve all directory names. No download or archive extraction is automated here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/IMAD/data/genimage/Stable Diffusion V1.4')  # edit to actual extracted root
OUTPUT_DIR = Path('/content/drive/MyDrive/IMAD/experiments/genimage_resnet18')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_ROOT.is_dir(), f'Dataset path does not exist: {DATA_ROOT}'
print('Dataset root:', DATA_ROOT)
print('Top-level entries:', sorted(p.name for p in DATA_ROOT.iterdir())[:30])


## Metadata-only inspection (no image decoding)
This scans paths and reports counts by generator, official split, and source class. It does not verify image integrity.


In [ ]:
subprocess.run([sys.executable, 'scripts/prepare_genimage.py', str(DATA_ROOT), '--metadata-only',
                '--inspection-output', str(OUTPUT_DIR / 'metadata_inspection.json')], check=True)


## Small image audit
The sample is stratified across train/validation × ai/nature and checks decoding, dimensions, exact hashes, and difference hashes. A sample is useful for path/decoder checks only; do not train or report this as the definitive dataset evaluation.


In [ ]:
SAMPLE_SIZE = 32
subprocess.run([sys.executable, 'scripts/prepare_genimage.py', str(DATA_ROOT), '--sample-size', str(SAMPLE_SIZE),
                '--seed', '42', '--inspection-output', str(OUTPUT_DIR / 'sample_inspection.json')], check=True)


## Full audit and manifest
This opens and hashes every image under the selected root. It may take a long time and needs temporary memory for the inventory. It refuses unrecognized layouts, corrupt files remain reported and are excluded, and duplicate groups across official train/val stop manifest creation for manual review. Equal SHA-256 or same-class dHash images stay together; dHash is a conservative duplicate cue, not proof of identical content.

Official validation remains validation. The script splits only official train groups into train and internal test using seed 42. It does not combine independent datasets or tune on external data.


In [ ]:
FULL_AUDIT = True
if FULL_AUDIT:
    subprocess.run([sys.executable, 'scripts/prepare_genimage.py', str(DATA_ROOT), '--seed', '42',
                    '--internal-test-fraction', '0.15',
                    '--inspection-output', str(OUTPUT_DIR / 'full_inspection.json'),
                    '--manifest-output', str(OUTPUT_DIR / 'split_manifest.csv')], check=True)
    manifest_path = OUTPUT_DIR / 'split_manifest.csv'


## GPU smoke fit, then full fit
The smoke run uses at most 16 items per class in train/validation/internal-test partitions for one epoch, but it only iterates train and validation. It does not open or evaluate the internal-test images.

After the smoke run completes, set `SMOKE_TEST = False` and `RUN_FULL_TRAINING = True`, then run the cell again. If Colab disconnects during a full run, set `RESUME = True`; checkpoints are written after each epoch. The internal test is evaluated only after the full run selects the best checkpoint by validation loss. Do not use test metrics to choose epochs, thresholds, or architectures.


In [ ]:
SMOKE_TEST = True
RUN_FULL_TRAINING = False
RESUME = False
if not torch.cuda.is_available() and RUN_FULL_TRAINING:
    raise RuntimeError('GPU not detected. Select a Colab GPU runtime before full training.')
base = [sys.executable, 'scripts/train_genimage_baseline.py', str(manifest_path),
        '--output-dir', str(OUTPUT_DIR / 'artifacts'), '--seed', '42', '--workers', '2']
if SMOKE_TEST:
    subprocess.run(base + ['--smoke-test', '--epochs', '1', '--batch-size', '8'], check=True)
if RUN_FULL_TRAINING:
    command = base + ['--epochs', '5', '--batch-size', '32']
    if RESUME: command.append('--resume')
    subprocess.run(command, check=True)


## Review saved results
Full-run artifacts include best and latest trusted checkpoints, label map, resolved configuration/runtime versions, train/validation history, internal-test metrics and predictions, and confusion/training plots. `validation_metrics` are tracked during training; `internal_test_metrics.json` is produced from the validation-selected checkpoint. Smoke runs save training history/checkpoints/curves only. Do not describe smoke outputs as model results. Copy artifacts back to the local project's ignored `models/` and `reports/experiments/` folders if desired.


In [ ]:
metrics_path = OUTPUT_DIR / 'artifacts' / 'internal_test_metrics.json'
if metrics_path.exists():
    print(metrics_path.read_text())
else:
    print('No internal-test metrics yet. Run the smoke or full-training cell first.')
